# Gemma3-1B 모델 pre-training을 위한 5/3/1 설명 데이터 생성
- OpenAI API 활용 : `GPT-4o-mini`

### 생성된 데이터 샘플
```py
{
    "question": "531 by jim wendler에 대해서 설명해줘.",
    "answer": "\"531\"은 Jim Wendler가 개발한 인기 있는 웨이트 트레이닝 프로그램입니다. 이 프로그램은 주로 근력 향상에 초점을 맞추고 있으며, 특히 중량 운동을 통해 강한 힘과 근육을 키우고자 하는 사람들에게 적합합니다. \"531\"의 이름은 훈련 방식에서 차용된 것으로, 주로 사용되는 반복 횟수와 세트를 나타냅니다.\n\n### 프로그램의 기본 구성\n1. **주요 운동**: 531 프로그램은 주요 운동으로 스쿼트, 벤치 프레스, 데드리프트, 오버헤드 프레스를 포함합니다.\n2. **주기**: 프로그램은 4주 주기로 구성되어 있으며, 각 주마다 중량이 증가합니다.\n   - 1주차: 3회 반복 (3RM)\n   - 2주차: 3회 반복 (3RM) + 1회 반복 (1RM)\n   - 3주차: 5회 반복 (5RM) + 3회 반복 (3RM) + 1회 반복 (1RM)\n   - 4주차: 'deload' 주로, 중량을 줄이고 회복을 목표로 합니다.\n\n### 중량 계산\n- 프로그램의 중량은 사용자의 1회 최대 중량(1RM)을 기준으로 계산됩니다. 각 운동의 목표 중량은 1RM의 일정 비율로 설정됩니다. 예를 들어, 90%의 중량에서 시작하여 점진적으로 중량을 증가시키는 방식입니다.\n\n### 특징\n- **단순성과 유연성**: 531은 매우 간단하게 설계되어 있으며, 개인의 목표와 체력 수준에 맞게 조정할 수 있습니다.\n- **보조 운동**: 주요 운동 외에도 보조 운동을 추가하여 전체적인 근력과 근육 발달을 도울 수 있습니다.\n- **프로그레시브 오버로드**: 점진적으로 중량을 늘려가는 방식으로, 지속적인 근력 향상을 목표로 합니다.\n\nJim Wendler의 531 프로그램은 많은 보디빌더와 파워리프터들 사이에서 인기가 있으며, 실용성과 효율성 덕분에 여러 운동 루틴에 통합되어 사용되기도 합니다."
},
```

In [2]:
from os import getenv
import os

import time

from dotenv import load_dotenv
from openai import OpenAI
import openai

import pandas as pd
import json

In [2]:
load_dotenv()
api_key = getenv("OPENAI_API_KEY")
client = openai.OpenAI(
    api_key = api_key
)

In [3]:
# 총 30개의 Seed 프롬프트
user_inputs = [
    "531 by jim wendler에 대해서 설명해줘.", # 기초적인 질문
    "5/3/1 운동 루틴은 어떻게 구성되어 있나요?",
    "헬스 입문자인데요, 531 프로그램이 뭔가요?",
    "요즘 531 루틴 한다고 들었는데, 그게 대체 뭐야?",
    "Jim Wendler의 5/3/1 프로그램에서 각 주차별 세트 구성은 어떻게 되나요?",
    "5/3/1 루틴이 다른 파워리프팅 프로그램이랑 비교해서 어떤 점이 좋아요?",
    "근력 향상 목적이면 531 루틴이 효과적인가요?",
    "531 프로그램으로 운동해보신 분들, 효과 어떤가요?",
    "제가 운동 경력이 별로 없는데, 5/3/1 루틴 시작해도 괜찮을까요?",
    "주 3회 운동 가능한데, 5/3/1 루틴으로 프로그램 짜면 어떻게 될까요?",
    "스쿼트 1RM이 140인데, 5/3/1 루틴에서 어느 정도 무게로 시작하나요?",
    "5/3/1에서 TM을 90%로 설정하는 이유가 뭔가요?", #자세한 질문
    "531 루틴의 조커 세트와 퍼스널 레코드(PR) 세트는 어떤 원리로 구성되나요?",
    "5/3/1을 진행하면서 볼륨 부족을 느낄 때 어떤 보조 루틴을 추가하는 게 좋을까요?",
    "Beyond 5/3/1에서 SST 방식은 어떻게 적용되나요?",
    "531 루틴에서 deload 주는 몇 주 차에 설정하는 게 일반적인가요?",
    "5/3/1 루틴을 오버헤드 프레스 강화 목적으로 최적화하려면 어떻게 구성해야 하나요?",
    "531 프로그램에서 각 리프트별 주간 볼륨이나 인텐시티 분포를 어떻게 분석하나요?",
    "웨이트 중급자가 531 루틴을 장기적으로 운영할 때 주기화는 어떻게 조절하나요?",
    "5/3/1 루틴을 벌크업 목적에 맞게 변형할 수 있는 방법이 있을까요?",
    "531에서 보조 운동은 어떤 기준으로 선택하고 주차별로 어떻게 조절해야 하나요?"
    "짐 웬들러 루틴이 그렇게 좋다던데, 어떤 방식이에요?", # 명칭이 다른 경우
    "웨이트 루틴 중에 531 방식이라는 거 있던데, 그거 설명 좀 해주세요.",
    "웬들러 프로그램이 근력 향상에 좋다고 들었어요. 주차별 구성은 어떻게 되나요?",
    "531 시스템? 그거 어떻게 시작해야 해요?",
    "헬스장 형이 알려준 웬들러 방식 루틴, 그거 정확히 뭔가요?",
    "그 유명한 5-3-1 루틴 구성 좀 알려주세요.",
    "웬들러 템플릿을 제 운동 스케줄에 적용하려면 어떻게 해야 하나요?",
    "근력 위주 루틴 중에 숫자로 된 거… 5 3 1 그거 있잖아요, 그건 어떻게 하는 거예요?",
    "짐 웬들러 방식 루틴에서 TM이란 게 나오던데, 그거 계산법이 어떻게 되죠?",
    "5, 3, 1로 점점 무게 올리는 루틴 있잖아요. 그 원리가 뭐예요?"
]

In [4]:
outputs = []
model_name = "gpt-4o-mini"

for index, user_input in enumerate(user_inputs):
    try:
        response = client.chat.completions.create(
            model=f"{model_name}",
            messages=[{"role":"user","content":[{"type":"text","text":f"{user_input}"}]}],
            temperature=0.8,
            top_p=0.9
        )
        
        result = response.choices[0].message.content
        outputs.append(result)
        time.sleep(3)
        print(f"index : {index} | ✅ Good")
    except Exception as e:
        outputs.append(e)
        print(f"index : {index} | ❌ ERROR : {e}")

index : 0 | ✅ Good
index : 1 | ✅ Good
index : 2 | ✅ Good
index : 3 | ✅ Good
index : 4 | ✅ Good
index : 5 | ✅ Good
index : 6 | ✅ Good
index : 7 | ✅ Good
index : 8 | ✅ Good
index : 9 | ✅ Good
index : 10 | ✅ Good
index : 11 | ✅ Good
index : 12 | ✅ Good
index : 13 | ✅ Good
index : 14 | ✅ Good
index : 15 | ✅ Good
index : 16 | ✅ Good
index : 17 | ✅ Good
index : 18 | ✅ Good
index : 19 | ✅ Good
index : 20 | ✅ Good
index : 21 | ✅ Good
index : 22 | ✅ Good
index : 23 | ✅ Good
index : 24 | ✅ Good
index : 25 | ✅ Good
index : 26 | ✅ Good
index : 27 | ✅ Good
index : 28 | ✅ Good
index : 29 | ✅ Good


In [5]:
df = pd.DataFrame()
df["inputs"] = user_inputs
df["outputs"] = outputs

In [6]:
df.to_excel(f"./dataset/pre-training/generated_data({model_name}).xlsx", index=False)

In [5]:
df = pd.read_excel(f"./dataset/pre-training/generated_data({model_name}).xlsx")

# Gemma3 포맷 변환 함수
def convert_to_qa_format(user_input, model_output):
    return {
        "question": user_input.strip(),
        "answer": model_output.strip()
    }

# 포맷 변환
qa_data = df.apply(
    lambda row: convert_to_qa_format(row["inputs"], row["outputs"]),
    axis=1
).tolist()

# JSON 저장
output_path = "./dataset/pre-training/gemma3_formatted_for_training.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(qa_data, f, ensure_ascii=False, indent=2)

print(f"Saved to: {output_path}")


Saved to: ./dataset/pre-training/gemma3_formatted_for_training.json
